In [1]:
# ✅ Balancing Dataset Function

from sklearn.utils import resample

def get_balanced_dataset():
    df = pd.read_csv("TweetSentiment.csv", encoding="ISO-8859-1")[["text", "sentiment"]]
    df.dropna(inplace=True)

    max_size = df["sentiment"].value_counts().max()
    df_balanced = pd.concat([
        resample(class_df, replace=True, n_samples=max_size, random_state=42)
        for _, class_df in df.groupby("sentiment")
    ])

    return df_balanced

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertModel
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=max_length, return_tensors='pt')
        self.labels = torch.tensor(labels.values)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

class BERT_RNN(nn.Module):
    def __init__(self, hidden_dim, output_dim, rnn_type='lstm', dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.rnn_type = rnn_type.lower()
        self.hidden_dim = hidden_dim

        if self.rnn_type == 'gru':
            self.rnn = nn.GRU(self.bert.config.hidden_size, hidden_dim, batch_first=True)
        else:
            self.rnn = nn.LSTM(self.bert.config.hidden_size, hidden_dim, batch_first=True)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = bert_outputs.last_hidden_state
        _, hidden = self.rnn(sequence_output)

        if self.rnn_type == 'lstm':
            hidden = hidden[0]

        output = self.fc(self.dropout(hidden[-1]))
        return output

def run_rnn_pipeline(df, title_prefix):
    df["label"] = df["sentiment"].astype("category").cat.codes
    num_classes = df["label"].nunique()
    
    X_train, X_test, y_train, y_test = train_test_split(
        df["text"], df["label"], test_size=0.2, stratify=df["label"], random_state=SEED
    )

    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    train_dataset = TweetDataset(X_train, y_train, tokenizer)
    test_dataset = TweetDataset(X_test, y_test, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32)

    model = BERT_RNN(hidden_dim=128, output_dim=num_classes, rnn_type='lstm').to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=2e-4)

    model.train()
    for epoch in range(3):
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(f"\n--- {title_prefix} BERT + RNN ---")
    print(classification_report(all_labels, all_preds, digits=4))
    ConfusionMatrixDisplay.from_predictions(all_labels, all_preds)
    plt.title(f"{title_prefix} BERT + RNN Confusion Matrix")
    plt.show()

df = get_balanced_dataset()

df_binary = df[df["sentiment"].isin(["positive", "negative"])].copy()
run_rnn_pipeline(df_binary, "Binary")

df_multiclass = df.copy()
run_rnn_pipeline(df_multiclass, "Multiclass")

In [ ]:
import random
from sklearn.metrics import accuracy_score
from copy import deepcopy

search_space = {
    "hidden_dim": [64, 128, 256],
    "dropout": [0.1, 0.3, 0.5],
    "lr": [1e-5, 5e-5, 1e-4, 5e-4],
    "rnn_type": ["lstm", "gru"],
    "batch_size": [16, 32],
    "epochs": [3, 5, 10]
}

def random_search(
    df, tokenizer, n_trials=10, max_length=64
):
    df["label"] = df["sentiment"].astype("category").cat.codes
    num_classes = df["label"].nunique()
    
    X_train, X_val, y_train, y_val = train_test_split(
        df["text"], df["label"], test_size=0.2, stratify=df["label"], random_state=SEED
    )
    
    best_score = 0
    best_params = None

    for trial in range(n_trials):
        params = {
            "hidden_dim": random.choice(search_space["hidden_dim"]),
            "dropout": random.choice(search_space["dropout"]),
            "lr": random.choice(search_space["lr"]),
            "rnn_type": random.choice(search_space["rnn_type"]),
            "batch_size": random.choice(search_space["batch_size"]),
            "epochs": random.choice(search_space["epochs"]),
        }

        print(f"\n🧪 Trial {trial+1}/{n_trials} with params: {params}")

        train_dataset = TweetDataset(X_train, y_train, tokenizer, max_length)
        val_dataset = TweetDataset(X_val, y_val, tokenizer, max_length)

        train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32)

        model = BERT_RNN(
            hidden_dim=params["hidden_dim"],
            output_dim=num_classes,
            rnn_type=params["rnn_type"],
            dropout=params["dropout"]
        ).to(device)

        optimizer = optim.Adam(model.parameters(), lr=params["lr"])
        criterion = nn.CrossEntropyLoss()

        model.train()
        for epoch in range(params["epochs"]):
            for batch in train_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                optimizer.zero_grad()
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

        model.eval()
        preds, true = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                outputs = model(input_ids, attention_mask)
                pred = torch.argmax(outputs, dim=1)
                preds.extend(pred.cpu().numpy())
                true.extend(labels.cpu().numpy())

        acc = accuracy_score(true, preds)
        print(f"Validation Accuracy: {acc:.4f}")

        if acc > best_score:
            best_score = acc
            best_params = deepcopy(params)

    print("\n✅ Best Parameters:")
    print(best_params)
    print(f"🎯 Best Validation Accuracy: {best_score:.4f}")
    return best_params